In [1]:
import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

Select required columns

In [3]:
data_path = "../data/ohlcv.parquet"
data = pd.read_parquet(data_path)

In [4]:
cols = ['date', 'co_name', 'open', 'high', 'low', 'close', 'volume']
data = data[cols]

In [5]:
data = data.sort_values(['date', 'co_name'], ignore_index=True)

Create daily return column (using close)

In [7]:
data = data.sort_values(['date', 'co_name'], ignore_index=True)

In [8]:
data['daily_return'] = data.groupby('co_name')['close'].pct_change()

In [9]:
data['prev_close'] = data.groupby('co_name')['close'].shift(1)

In [10]:
data['prev_date'] = data.groupby('co_name')['date'].shift(1)

In [11]:
data['days_diff'] = (data['date'] - data['prev_date']).dt.days

Create quarter column

In [6]:
def assign_quarter(date):
    month = date.month
    day = date.day
    year = date.year
    
    # 15th Feb to 30th May
    if (month == 2 and day >= 15) or (month in [3, 4]) or (month == 5 and day <= 30):
        return f'{year}02'
    # 31st May to 14th August
    elif (month == 5 and day >= 31) or (month in [6, 7]) or (month == 8 and day <= 14):
        return f'{year}05'
    # 15th August to 14th November
    elif (month == 8 and day >= 15) or (month in [9, 10]) or (month == 11 and day <= 14):
        return f'{year}08'
    # 15th November to 14th Feb
    else:
        # If in Jan or early Feb (before 15th), use previous year
        if month in [1] or (month == 2 and day < 15):
            return f'{year - 1}11'
        # If in Nov or Dec, use current year
        else:
            return f'{year}11'

data['quarter'] = data['date'].apply(assign_quarter)

In [7]:
data.to_parquet('../data/ohlcv2.parquet', index=False)

Compute average daily return - averaged over the quarter

In [10]:
avg_daily_return = data.groupby(['co_name', 'quarter'])['daily_return'].mean().reset_index()
avg_daily_return = avg_daily_return.rename(columns={'daily_return': 'avg_daily_return'})

For every company, for every quarter, get the max high price

In [11]:
maximum_quarterly_high_price = data.groupby(['co_name', 'quarter'])['high'].max().reset_index()
maximum_quarterly_high_price = maximum_quarterly_high_price.rename(columns={'high': 'quarter_high'})

For every company, for every quarter, the mean of the five largest highs

In [10]:
avg_top5_high = (
    data.groupby(['co_name', 'quarter'])['high']
    .apply(lambda x: x.nlargest(5).mean())
    .reset_index()
    .rename(columns={'high': 'quarter_high_5'})
)

For every quarter and every company, the mean of the five largest closes

In [11]:
avg_top5_close = (
    data.groupby(['co_name', 'quarter'])['close']
    .apply(lambda x: x.nlargest(5).mean())
    .reset_index()
    .rename(columns={'close': 'quarter_close_5'})
)

For every quarter and every company, the mean of the first three closes

In [12]:
data = data.sort_values(by=['date', 'co_name'], ignore_index=True)

avg_first3_close = (
    data.groupby(['co_name', 'quarter'])
    .apply(lambda x: x.nsmallest(3, 'date')['close'].mean(), include_groups=False)
    .reset_index()
    .rename(columns={0: 'avg_first3_close'})
)

Create Targets

In [13]:
df = avg_first3_close.merge(avg_top5_close, on=['co_name', 'quarter']).merge(avg_top5_high, on=['co_name', 'quarter'])

In [14]:
df.columns

Index(['co_name', 'quarter', 'avg_first3_close', 'quarter_close_5',
       'quarter_high_5'],
      dtype='object')

In [15]:
df['high5-close3'] = (df['quarter_high_5'] - df['avg_first3_close']) / df['avg_first3_close']
df['close5-close3'] = (df['quarter_close_5'] - df['avg_first3_close']) / df['avg_first3_close']

In [16]:
target1 = df.copy()
target1['percentile'] = target1.groupby('quarter')['high5-close3'].rank(pct=True)
target1['target'] = (target1['percentile'] > 0.6).astype(int)

In [17]:
target2 = df.copy()
target2['percentile'] = target2.groupby('quarter')['close5-close3'].rank(pct=True)
target2['target'] = (target2['percentile'] > 0.6).astype(int)

In [18]:
target3 = df.copy()
target3['percentile'] = target3.groupby('quarter')['high5-close3'].rank(pct=True)
target3['target'] = (target3['percentile'] > 0.8).astype(int)

In [19]:
target4 = df.copy()
target4['percentile'] = target4.groupby('quarter')['close5-close3'].rank(pct=True)
target4['target'] = (target4['percentile'] > 0.8).astype(int)

Compute quarterly volatility of stocks (std dev of daily returns over the quarter)

In [20]:
# Calculate quarterly volatility (standard deviation of daily returns)
quarterly_volatility = data.groupby(['co_name', 'quarter'])['daily_return'].std().reset_index()
quarterly_volatility = quarterly_volatility.rename(columns={'daily_return': 'volatility'})

In [21]:
# Merge volatility into the main dataframe
df = df.merge(quarterly_volatility, on=['co_name', 'quarter'])

In [22]:
# Create Efficiency Metrics
df['efficiency_high'] = df['high5-close3'] / df['volatility']
df['efficiency_close'] = df['close5-close3'] / df['volatility']

In [23]:
# Create Target 5 (Efficiency based on Highs)
target5 = df.copy()
target5['percentile'] = target5.groupby('quarter')['efficiency_high'].rank(pct=True)
target5['target'] = (target5['percentile'] > 0.8).astype(int)

In [24]:
# Create Target 6 (Efficiency based on Closes)
target6 = df.copy()
target6['percentile'] = target6.groupby('quarter')['efficiency_close'].rank(pct=True)
target6['target'] = (target6['percentile'] > 0.8).astype(int)

Merge all targets into a single dataframe

In [25]:
# Extract relevant columns and rename target column
t1 = target1[['co_name', 'quarter', 'target']].rename(columns={'target': 'target1'})
t2 = target2[['co_name', 'quarter', 'target']].rename(columns={'target': 'target2'})
t3 = target3[['co_name', 'quarter', 'target']].rename(columns={'target': 'target3'})
t4 = target4[['co_name', 'quarter', 'target']].rename(columns={'target': 'target4'})
t5 = target5[['co_name', 'quarter', 'target']].rename(columns={'target': 'target5'})
t6 = target6[['co_name', 'quarter', 'target']].rename(columns={'target': 'target6'})

# Merge all targets
all_targets = t1.merge(t2, on=['co_name', 'quarter']) \
                .merge(t3, on=['co_name', 'quarter']) \
                .merge(t4, on=['co_name', 'quarter']) \
                .merge(t5, on=['co_name', 'quarter']) \
                .merge(t6, on=['co_name', 'quarter'])

In [26]:
all_targets.to_parquet('../data/targets.parquet', index=False)